# Transformer-Based LLM Chatbot using Hugging Face Transformers and Gradio

**Vishal S**  `24BAD130`

## Part A — Environment Setup and Model Selection

Install the libraries needed for the pretrained model, PyTorch, and Gradio interface.

In [ ]:
# Vishal S 24BAD130
%pip -q install -U transformers accelerate gradio

In [ ]:

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Select GPU when available; otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)
model.to(device)
model.eval()

print("Model and tokenizer loaded successfully!")

Using device: cpu
Loading model: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In [ ]:
# Simple first test to confirm that the model generates a response

test_messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer clearly and briefly."},
    {"role": "user", "content": "What is artificial intelligence?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
first_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Prompt: What is artificial intelligence?")
print("Response:", first_response)

Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, language understanding, and decision-making. It encompasses various subfields like machine learning, natural language processing, computer vision, robotics, autonomous vehicles, and more. AI aims to make machines that can think, act, learn, and communicate in ways similar to humans.


## Part B — Prompt Handling and Response Generation

The function below accepts a prompt, includes recent conversation history, generates a response, checks for empty input, and catches errors.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful AI chatbot for a student. "
    "Answer in simple, clear language. If you are unsure, say so."
)

def chatbot_response(message, history=None):
    # Handle empty input
    if not message or not message.strip():
        return "Please enter a message."

    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        # Include recent conversation history when available
        if history:
            for item in history[-6:]:
                if isinstance(item, dict):
                    role = item.get("role")
                    content = item.get("content", "")
                    if role in ("user", "assistant") and isinstance(content, str):
                        messages.append({"role": role, "content": content})
                elif isinstance(item, (list, tuple)) and len(item) == 2:
                    user_text, assistant_text = item
                    if user_text:
                        messages.append({"role": "user", "content": str(user_text)})
                    if assistant_text:
                        messages.append({"role": "assistant", "content": str(assistant_text)})

        messages.append({"role": "user", "content": message.strip()})

        # Format messages using the model's chat template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            truncation=True,
            max_length=2048
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens, not the prompt
        prompt_length = inputs["input_ids"].shape[-1]
        response = tokenizer.decode(
            output_ids[0][prompt_length:],
            skip_special_tokens=True
        ).strip()

        if not response:
            return "Sorry, I could not generate a response. Please try another prompt."

        return response

    except Exception as error:
        print("Chatbot error:", error)
        return "Sorry, an error occurred while generating the response. Please try again."

print("Chatbot response function is ready.")

Chatbot response function is ready.


In [ ]:
# Test at least five different prompts

test_prompts = [
    "What is artificial intelligence?",
    "Explain Python in simple words.",
    "What is a Transformer in machine learning?",
    "Give me three tips to prepare for an exam.",
    "Write a short motivational message for a student."
]

for number, prompt in enumerate(test_prompts, start=1):
    answer = chatbot_response(prompt, history=[])
    print(f"TEST {number}")
    print("Prompt:", prompt)
    print("Response:", answer)
    print("-" * 70)

TEST 1
Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) is a broad field that focuses on creating machines that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and learning from experience. Here are some key aspects of AI:

1. **Machine Learning**: This involves algorithms that learn and improve over time based on data, allowing machines to adapt and become more efficient.

2. **Deep Learning**: A subset of machine learning where the model uses multiple layers to process information, often involving neural networks with thousands or millions of neurons.

3. **Natural Language Processing (NLP)**:
----------------------------------------------------------------------
TEST 2
Prompt: Explain Python in simple words.
Response: Sure! Here's a simple explanation of Python:

Python is like a magical code book that helps people write computer programs. It's used to make things work automat

In [ ]:
# Test empty input and a follow-up conversation

print("Empty input test:")
print(chatbot_response("   "))

sample_history = [
    {"role": "user", "content": "My name is Sam."},
    {"role": "assistant", "content": "Nice to meet you, Sam."}
]

print("\nConversation history test:")
print("Prompt: What is my name?")
print("Response:", chatbot_response("What is my name?", history=sample_history))

Empty input test:
Please enter a message.

Conversation history test:
Prompt: What is my name?
Response: Your name is Sam.


## Part C — Build the Chatbot Interface

Run the following cell to create an interactive Gradio chatbot. It displays messages, passes conversation history to the response function, and provides a clear-chat control.

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Transformer-AI Chatbot",
    description=(
        "A simple chatbot powered by Hugging Face Transformers and "
        "Qwen2.5-0.5B-Instruct. Enter a prompt below to begin."
    ),
    chatbot=gr.Chatbot(height=420),
    textbox=gr.Textbox(
        placeholder="Type your message and press Enter...",
        label="Your message"
    ),
    examples=[
        "What is artificial intelligence?",
        "Explain Python in simple words.",
        "What is a Transformer model?",
        "Give me three study tips."
    ]
)

print("Gradio interface created. Launch it in the next cell.")

Gradio interface created. Launch it in the next cell.


In [ ]:
# In Google Colab, share=True provides a temporary public demo URL.
# Do not use share=True for private or sensitive conversations.

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a871a336b3cf098f8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Part D — Testing, Evaluation, and Deployment

Use the next cell to run a small functional test. Manually review each answer for relevance, correctness, readability, and consistency. A response being generated does not guarantee that it is factually correct.

In [ ]:
evaluation_cases = {
    "Factual": "What is machine learning?",
    "Conversational": "Hello! How are you?",
    "Task-oriented": "List three ways to manage study time."
}

for category, prompt in evaluation_cases.items():
    answer = chatbot_response(prompt, history=[])
    print("Category:", category)
    print("Prompt:", prompt)
    print("Response:", answer)
    print("Manual review needed: Check accuracy and relevance.")
    print("-" * 70)

Category: Factual
Prompt: What is machine learning?
Response: Machine learning is a type of artificial intelligence (AI) that allows computers to learn and improve from experience without being explicitly programmed. It involves using algorithms to analyze data and make decisions or predictions based on patterns found in that data. Essentially, it's about teaching machines how to perform tasks by showing them examples of correct outcomes. Machine learning can be applied in many areas like image recognition, natural language processing, and predictive analytics.
Manual review needed: Check accuracy and relevance.
----------------------------------------------------------------------
Category: Conversational
Prompt: Hello! How are you?
Response: Hello! I'm just a simple assistant here to help with anything else you need help with. How can I assist you today?
Manual review needed: Check accuracy and relevance.
----------------------------------------------------------------------
Category

In [ ]:
# Create a requirements.txt file for local use or deployment

requirements_text = """torch
transformers
accelerate
gradio
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("requirements.txt created.")
print(requirements_text)

requirements.txt created.
torch
transformers
accelerate
gradio



### Conclusion
A prompt-based chatbot was built using a pretrained Transformer language model from Hugging Face, with response generation, basic error handling, conversation history, a Gradio interface, and functional testing. Model responses can still be inaccurate, repetitive, or biased, so they should be checked before relying on them.
